# 02 — Genie Agents

**Time: about 12 minutes.**
Covers: Genie Agents (formerly Genie Spaces) · Data sources · Text instructions · Example SQL · Benchmarks · Scoping

### What you will be able to do afterwards

- Create a Genie Agent on top of a metric view and configure it with instructions, example SQL and a benchmark.
- Explain why a tightly scoped agent (one bounded domain) answers more accurately than one pointed at every table.
- Read an agent's configuration back through the Genie API, the way the checks do.

## What a Genie Agent is

A Genie Agent is a domain-specific, natural-language chat interface over data you choose. Its author
(you, the data engineer) controls everything the LLM gets to work with:

| Part | What it is | Where |
|---|---|---|
| Data sources | Up to 50 UC tables, views or metric views. Genie only queries what is attached. | Configure > Data |
| Text instructions | Business rules in plain language. One text block per agent. | Configure > Instructions |
| Example SQL | Question + correct SQL pairs that Genie imitates. | Configure > Instructions > SQL Queries |
| Benchmarks | Question + correct SQL used to *score* the agent. Not shown to Genie as hints. | Benchmarks |
| Warehouse | Pro or serverless SQL warehouse that runs Genie's SQL. | Configure > Settings |

**Scoping drives accuracy.** Every extra table is another place Genie can find a plausible-looking
`revenue` column or a join at the wrong grain. A metric view gives the agent one governed definition
per metric; this agent gets the metric view and one small lookup, nothing else.

**Requirements** (if any is missing, Genie Agents are hidden or empty; tell your instructor):
the Databricks SQL entitlement, CAN USE on a pro or serverless SQL warehouse, and partner-powered AI
features enabled for the account and workspace.

In [ ]:
from helpers import utils, genie_agents, semantic_model

cfg = utils.use_lab_schema()
spark = utils.spark
print(f"Working in {cfg['catalog']}.{cfg['schema']}")

## Step 1 — Prepare what the UI needs

In [ ]:
assert semantic_model.get_definition(), "The metric view is missing. Finish 01 first."
print("Data sources to attach (and nothing else):")
print(f"  metric view   {utils.fq_plain(utils.METRIC_VIEW)}")
print(f"  lookup        {utils.fq_plain('lab_region_targets')}")
print("\nSQL warehouses you can see (pick a serverless one if listed):")
for wh in genie_agents.find_warehouses()[:8]:
    print(f"  {wh['name']:40s} {wh['type']:12s} {wh['state']:10s} {wh['id']}")

In [ ]:
genie_agents.print_agent_kit()

## Step 2 — Build the agent in the UI

Labels move between releases; if a name below differs slightly, look for the nearest equivalent.

1. In the left sidebar click **Genie Agents** (some workspaces still show **Genie**), then **New**.
2. In the data picker select **only** the metric view and `lab_region_targets` from Step 1. Click **Create**.
3. **Genie Code** may open automatically and suggest context (descriptions, example queries). Read
   the suggestions. Accepting descriptions is fine; do **not** accept anything that adds other tables.
4. **Configure > Settings**: set the **Title** exactly as printed (the checks find your agent by it),
   pick the **default warehouse**, paste the **Description**.
5. **Configure > Instructions > Text**: paste the whole TEXT INSTRUCTIONS block. An agent holds one
   text instruction, so all the rules go into that one block.
6. **Configure > Instructions > SQL Queries > Add**: paste the example question and its SQL.
7. Open **Benchmarks** > **Add benchmark**: paste the benchmark question and its SQL answer.
   Optionally click **Run** to score the agent.
8. In the chat, ask *What was our revenue in the last 90 days?* and then *Which product category had
   the highest revenue?* Open the generated SQL (**Show code** / **View SQL**) for each. Both should
   query `sales_metrics` with `MEASURE(net_revenue)`.

## Step 3 — Register the agent

Paste the agent ID (the 32-character hex string in its URL), or leave it empty to look it up by the
title you set in step 4.

In [ ]:
AGENT_ID = ""  # paste the ID, or leave empty to auto-discover by title

genie_agents.register("metric_view_agent", AGENT_ID)

## Step 4 — Read the configuration back

Everything you clicked is a Databricks object. `GET /api/2.0/genie/spaces/{id}?include_serialized_space=true`
returns it as JSON; the checks use exactly this.

In [ ]:
genie_agents.describe(genie_agents.registered_id("metric_view_agent"))

**What the checks verify, and what they cannot**

| Verified from the API and the data | Not verifiable, reviewed by your instructor |
|---|---|
| The agent exists, has a warehouse, and you can read it | Whether your instructions are clear and complete |
| Its data sources are your metric view (+ lookup) and no raw fact views | Whether Genie's *chat* answers are good |
| A text instruction mentions revenue | How it handles questions nobody anticipated |
| An example SQL queries the metric view with `MEASURE()` | |
| The benchmark SQL, **executed against the data**, returns the governed number | |

## Step 5 (optional) — Ask through the Conversation API

The same agent is reachable from code, which is how it gets embedded in apps and other agents.

In [ ]:
r = genie_agents.ask(genie_agents.registered_id("metric_view_agent"), "What was our revenue in the last 90 days?")
print(r["status"], r["first_value"])
print(r["sql"])

---

## Challenge 2

Add a second **benchmark** to your agent:

*Which region is furthest below its 90-day revenue target?*

The question must contain the word **target**. Its SQL answer must return the region furthest below
target in the first row (the region column can be anywhere in that row). Metric views cannot be
joined directly: aggregate the metric view in a CTE, then join `lab_region_targets` on `region`.

Write and test the SQL here first, then paste it into **Benchmarks > Add benchmark**. Adding it as an
example SQL query as well is a good idea; the check only looks at benchmarks.

**TO DO**

In [ ]:
challenge_sql = f"""
-- TO DO: the region furthest below its 90-day net revenue target must be in the first row.
-- Metric view: {utils.fq_plain(utils.METRIC_VIEW)}   Targets: {utils.fq_plain('lab_region_targets')}
SELECT 'TO DO' AS region
"""
display(spark.sql(challenge_sql))

## Checks

In [ ]:
from helpers import test_runner
test_runner.run("02-genie-agents")